In [ ]:
# Configuración COSMIC v0.0.1 - Estructura Modular
import sys
from pathlib import Path

# Configuración automática de rutas relativas
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parents[3]  # Tres niveles arriba desde data/test/NGC6383/

# Agregar al path si no está
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"🌟 COSMIC v0.0.1 - NGC6383 Preprocessing")
print(f"📁 Directorio actual: {CURRENT_DIR}")
print(f"📁 Proyecto: {PROJECT_ROOT}")

# Verificar instalación de COSMIC
try:
    import cosmic
    print("COSMIC modular disponible")
except ImportError:
    print("Instalando COSMIC...")
    import os
    os.system(f"pip install -e {PROJECT_ROOT}")
    import cosmic
    print("COSMIC instalado")

In [ ]:
%config InlineBackend.figure_format = 'retina'

In [ ]:
import os
import glob
import dill

from cosmic.preprocess.preprocessor import DataPreprocessor
from cosmic.io.loader import DataLoader
from cosmic.core.clustering import Clustering

BASE_DIR = "../data"

def find_ecsv(folder: str) -> str | None:
    matches = glob.glob(os.path.join(folder, "*-result.ecsv"))
    return matches[0] if matches else None

In [ ]:
FOLDER = os.path.join(BASE_DIR, "40")
ecsv    = find_ecsv(FOLDER)
out_ecsv = os.path.join(FOLDER, "clustering_results.ecsv")
out_dill = os.path.join(FOLDER, "clustering_results.dill")

print(f"Input : {ecsv}")
print(f"Output: {out_ecsv}")

In [ ]:
# ── Preprocessing ────────────────────────────────────────────────────────────
loader = DataLoader(ecsv)
data = loader.load_data(
    systems=['Gaia', 'TMASS', 'WISE'],
    include_distances=['geometric'],
    include_zp_cols=True,
    include_flux_errors=True,
    fidelity='fidelity_v2'
)

pre = DataPreprocessor(data)
pre.rename_columns()
pre.drop_invalid_sources()
pre.fill_missing_values()
pre.apply_zero_point_correction()
pre.correct_proper_motion()
pre.add_photometric_errors()
good_data, bad_data = pre.filter_data(fidelity_threshold=0.5)

print(f"Good: {len(good_data)}  Bad: {len(bad_data)}")

In [ ]:
# ── Clustering ───────────────────────────────────────────────────────────────
clust = Clustering(good_data, bad_data)
clust.search_pseudoprobability(
    columns=["pmra", "pmdec"],
    min_cluster_size_samples=range(10, 70),
    probability_threshold=0.5,
    min_cluster_members=200,
    max_cluster_members=1000,
    hdbscan_kwargs={
        "cluster_selection_method": "leaf",
        "allow_single_cluster": True,
    }
)
print("Best mcs :", clust.best_params_["min_cluster_size"])
print("λ        :", clust.best_score_)
print("Members  :", clust.pseudoprobability_selected_["desired_len"])

In [ ]:
# ── Evaluación ───────────────────────────────────────────────────────────────
clust.clustering_statistics()
clust.get_cluster_summary(include_noise=True)
clust.plot_cluster_members(show_outliers=False)
clust.plot_pm_scatter(show_outliers=False)
clust.plot_mcs_sweep()

In [ ]:
# ── Save ─────────────────────────────────────────────────────────────────────
clust.save_results(out_ecsv, format='ascii.ecsv')
with open(out_dill, "wb") as f:
    dill.dump(clust, f, protocol=dill.HIGHEST_PROTOCOL)
print(f"Saved: {out_ecsv}")